In [0]:
CREATE SCHEMA IF NOT EXISTS mvp;

In [0]:
CREATE TABLE mvp.shows_interpretes AS
SELECT
    COD_SHOW,

    NVL(
        NULLIF(UPPER(TRIM(DSC_SHOW)), ''),
        'EM CONFIRMAÇÃO'
    ) AS DSC_SHOW,

    DATA,

    UPPER(TRIM(LUGAR)) AS LUGAR,

    UPPER(
        CASE SGL_ESTADO
            WHEN 'AC' THEN 'Acre'
            WHEN 'AL' THEN 'Alagoas'
            WHEN 'AP' THEN 'Amapá'
            WHEN 'AM' THEN 'Amazonas'
            WHEN 'BA' THEN 'Bahia'
            WHEN 'CE' THEN 'Ceará'
            WHEN 'DF' THEN 'Distrito Federal'
            WHEN 'ES' THEN 'Espírito Santo'
            WHEN 'GO' THEN 'Goiás'
            WHEN 'MA' THEN 'Maranhão'
            WHEN 'MT' THEN 'Mato Grosso'
            WHEN 'MS' THEN 'Mato Grosso do Sul'
            WHEN 'MG' THEN 'Minas Gerais'
            WHEN 'PA' THEN 'Pará'
            WHEN 'PB' THEN 'Paraíba'
            WHEN 'PR' THEN 'Paraná'
            WHEN 'PE' THEN 'Pernambuco'
            WHEN 'PI' THEN 'Piauí'
            WHEN 'RJ' THEN 'Rio de Janeiro'
            WHEN 'RN' THEN 'Rio Grande do Norte'
            WHEN 'RS' THEN 'Rio Grande do Sul'
            WHEN 'RO' THEN 'Rondônia'
            WHEN 'RR' THEN 'Roraima'
            WHEN 'SC' THEN 'Santa Catarina'
            WHEN 'SP' THEN 'São Paulo'
            WHEN 'SE' THEN 'Sergipe'
            WHEN 'TO' THEN 'Tocantins'
            ELSE UPPER(TRIM(SGL_ESTADO))
        END
    ) AS SGL_ESTADO,

    QTD_INTER,
    COD_SHOW_INTERPRETE,

    NVL(
        NULLIF(UPPER(TRIM(NOME_INTERPRETE)), ''),
        'EM CONFIRMAÇÃO'
    ) AS NOME_INTERPRETE,

    CAST(
        REPLACE(TRIM(VALOR_RECEBIDO), ',', '.') AS DECIMAL(10,2)
    ) AS VALOR_RECEBIDO,

    NVL(
        NULLIF(UPPER(TRIM(TITULO_MUSICA)), ''),
        'EM CONFIRMAÇÃO'
    ) AS TITULO_MUSICA

FROM dadosgerais.shows_interpretes;


In [0]:
-- Quais lugares (estabelecimentos) ocorreram mais shows e estado --
SELECT
  I.LUGAR,
  I.SGL_ESTADO AS ESTADO,
  COUNT(*)
FROM
  mvp.shows_interpretes I
GROUP BY
  I.LUGAR,  I.SGL_ESTADO
ORDER BY
  COUNT(*) DESC

In [0]:
-- Quais estados ocorreram mais shows --
SELECT
  I.SGL_ESTADO AS ESTADO,
  COUNT(*)
FROM
  mvp.shows_interpretes I
GROUP BY
  I.SGL_ESTADO
ORDER BY
  COUNT(*) DESC

In [0]:
-- Volume de dinheiro (pagamento) de shows por estado --
SELECT TOTAL.sgl_estado as ESTADO, sum(TOTAL.TOTAL) AS TOTAL FROM (
  SELECT
    I.SGL_ESTADO,
    DADOS.valor_recebido AS TOTAL
  FROM
    mvp.shows_interpretes I,
    (
      SELECT DISTINCT
        INTER.COD_SHOW,
        INTER.COD_SHOW_INTERPRETE,
        INTER.VALOR_RECEBIDO,
        INTER.SGL_ESTADO
      FROM
        mvp.shows_interpretes INTER
    
    ) DADOS
  WHERE
    1 = 1
    AND I.COD_SHOW = DADOS.cod_show
    AND I.COD_SHOW_INTERPRETE = DADOS.COD_SHOW_INTERPRETE
    AND I.SGL_ESTADO = DADOS.SGL_ESTADO
  GROUP BY
      I.SGL_ESTADO, DADOS.valor_recebido
) TOTAL
GROUP BY TOTAL.sgl_estado ORDER BY sum(TOTAL.TOTAL) DESC

In [0]:
-- Quais intérpretes lucraram mais com shows --
SELECT
  DADOS.nome_interprete, sum(DADOS.valor_recebido) AS TOTAL
FROM
  (
    SELECT DISTINCT
      INTER.NOME_INTERPRETE,
      INTER.VALOR_RECEBIDO
    FROM
      mvp.shows_interpretes INTER
    WHERE
      1 = 1
  ) DADOS
  GROUP BY
    DADOS.nome_interprete
  ORDER BY
    sum(DADOS.valor_recebido) DESC

In [0]:
-- Quais intérpretes participaram de mais shows --
SELECT
  DADOS.nome_interprete, count(DADOS.cod_show) AS NUM_SHOW
FROM
  (   SELECT DISTINCT
      I.NOME_INTERPRETE, I.COD_SHOW
    FROM
      mvp.shows_interpretes I
    WHERE
      1 = 1 ) DADOS
  GROUP BY
    DADOS.NOME_INTERPRETE
  ORDER BY
   count(DADOS.cod_show) DESC

In [0]:
-- Quais shows tiveram mais músicas e onde ocorreram--
select
  I.DSC_SHOW,
  count(I.TITULO_MUSICA) AS NUM_MUSICAS,
  I.SGL_ESTADO AS ESTADO
from
  mvp.shows_interpretes I
group by
  I.DSC_SHOW, I.SGL_ESTADO
order by
  count(I.TITULO_MUSICA) desc

In [0]:
-- Quais intérpretes cantam mais músicas --
SELECT
  DADOS.nome_interprete, count(DADOS.titulo_musica) AS NUM_MUSICAS
FROM
  (   SELECT DISTINCT
      I.NOME_INTERPRETE AS NOME_INTERPRETE,
      I.TITULO_MUSICA AS TITULO_MUSICA
    FROM
      mvp.shows_interpretes I
    WHERE
      1 = 1
  ) DADOS
  GROUP BY
    DADOS.NOME_INTERPRETE
  ORDER BY
   count(DADOS.titulo_musica) DESC